In [1]:
import regex as re
from typing import List, Tuple, Dict

class BPETokenizer():
    def __init__(self):
        self.merges = {}  # (int, int) -> int
        self.id_to_text = {}  # int -> bytes
        self.text_to_id = {}

    def train(self, text, vocab_size, verbose=False):
        # 初步切分
        unique_chrs = sorted(list(set(list(text))))
        
        # 合并次数
        num_merges = vocab_size - len(unique_chrs)

        merges = {}
        
        #初始化词典
        id_to_char = {idx: ch for idx, ch in enumerate(unique_chrs)}
        char_to_id = {ch: idx for idx, ch in id_to_char.items()}
        if verbose:
            print("original char to id:")
            print(char_to_id)
        
        # ID化词典里的词
        ids = [char_to_id[ch] for ch in text]

        idx = len(id_to_char) - 1
        for i in range(num_merges):
            if len(ids) == 1:
                break
            # 统计相邻的id对出现的次数
            stats = self.stats(ids)
            if verbose:
                stats_text = {(id_to_char[pair[0]], id_to_char[pair[1]]):st for pair, st in stats.items()}
                print(f'{i+1}th iteration:')
                print(ids)
                print(stats_text)
            # 找到出现次数最多的id对
            pair = max(stats, key=stats.get)
            # 给出现次数最多的id对赋予新的id
            idx = idx + 1
            # 替换
            ids = self.merge_ids(ids, pair, idx)
            # 保存merge信息
            merges[pair] = idx
            id_to_char[idx] = id_to_char[pair[0]] + id_to_char[pair[1]]

            if verbose:
                print(f"merge {id_to_char[pair[0]], id_to_char[pair[1]]}->{id_to_char[idx]} at he {i + 1}th iteration")
                print("after merge, the vocabulary is")
                print(id_to_char)

        self.merges = merges
        self.id_to_text = id_to_char
        self.text_to_id = char_to_id

    def encode(self, text: str):
        ids = [self.text_to_id[c] for c in text]
        while len(ids) >= 2:
            stats = self.stats(ids)
            # 寻找可以融合的最小id。对于每个相邻对，merges里面找不到的时候认为对应的id为无穷大
            # 因为我们train的时候id就是从小到大生成的，所以这里也按照一样的顺序执行
            pair = min(stats, key=lambda p: self.merges.get(p, float('inf')))
            if pair not in self.merges:
                break
            idx = self.merges[pair]
            ids = self.merge_ids(ids, pair, idx)
        return ids

    def decode(self, ids):
        text = "".join(self.id_to_text[idx] for idx in ids)
        return text

    def stats(self, ids: List[int], counts = None):
        '''
        统计相邻ID出现的次数
        :param ids: 
        :param counts: 
        :return: 
        '''
        counts = {} if counts is None else counts
        for item in zip(ids, ids[1:]):
            counts[item] = counts.get(item, 0) + 1
        return counts

    def merge_ids(self, ids: List[int], pair: Tuple[int, int], idx: int) -> List[int]:
        '''
        更新待处理的文本ids
        :param ids: 
        :param pair: 
        :param idx: 
        :return: 
        '''
        newids = []
        i = 0
        while i < len(ids):
            if ids[i] == pair[0] and i < len(ids) - 1 and ids[i + 1] == pair[1]:
                newids.append(idx)
                i += 2
            else:
                newids.append(ids[i])
                i += 1
        return newids

In [2]:
train_text = """hello, this is a training text. The tokenizer will split the text into words and assign an id to each word. This is a fantastic world.
"""

tokenizer = BPETokenizer()
tokenizer.train(train_text, vocab_size=64, verbose=False)
print(tokenizer.id_to_text)
text = "hello, world"
ids = tokenizer.encode(text)
print(ids)
print(tokenizer.decode(ids))

{0: '\n', 1: ' ', 2: ',', 3: '.', 4: 'T', 5: 'a', 6: 'c', 7: 'd', 8: 'e', 9: 'f', 10: 'g', 11: 'h', 12: 'i', 13: 'k', 14: 'l', 15: 'n', 16: 'o', 17: 'p', 18: 'r', 19: 's', 20: 't', 21: 'w', 22: 'x', 23: 'z', 24: ' t', 25: 's ', 26: 'is ', 27: ' w', 28: 'he', 29: 'in', 30: ' wo', 31: ' wor', 32: 'an', 33: 'll', 34: 'his ', 35: 'his is ', 36: 'his is a', 37: ' te', 38: ' tex', 39: ' text', 40: '. ', 41: '. T', 42: ' to', 43: ' word', 44: 'as', 45: 'hell', 46: 'hello', 47: 'hello,', 48: 'hello, t', 49: 'hello, this is a', 50: 'hello, this is a t', 51: 'hello, this is a tr', 52: 'hello, this is a tra', 53: 'hello, this is a train', 54: 'hello, this is a trainin', 55: 'hello, this is a training', 56: 'hello, this is a training text', 57: 'hello, this is a training text. T', 58: 'hello, this is a training text. The', 59: 'hello, this is a training text. The to', 60: 'hello, this is a training text. The tok', 61: 'hello, this is a training text. The toke', 62: 'hello, this is a training text.

In [ ]:
'''
1. 字典的大小在数据比较少的时候很关键。我们上面的输入不长，但是字典大小设置为了64，最终出现了句子级别的id。
2. 直接对输入的裸字符进行编码并不是特别合适，上面就出现了\n开始的若干编码。
'''